# Oráculo normativo do TJPI no Google Colab

Este notebook usa as mesmas normas do [site do Oráculo](https://carlos-mendes-maestria.github.io/oraculo-normativo/), mas a resposta é escrita por uma **IA gratuita que roda aqui mesmo, no computador do Google**, sem ChatGPT nem Gemini.

**Como usar**

1. No menu, abra **Ambiente de execução › Alterar o tipo de ambiente de execução**, escolha **GPU T4** e salve.
2. Clique no botão ▶ da etapa 1 e aguarde. Se aparecer um aviso sobre o notebook não ser do Google, escolha **Executar assim mesmo**.
3. Clique no ▶ da etapa 2 e aguarde.
4. Na etapa 3, preencha o **tema** e, se quiser, a **pergunta**, e clique no ▶. Repita esta etapa quantas vezes quiser.

**Antes de usar**

- **Tamanho:** a etapa 1 baixa cerca de 8 GB, mas no computador do Google, não no seu. Leva de 5 a 10 minutos.
- **Sessão temporária:** a versão gratuita do Colab tem limite de uso. Ao fechar ou expirar, é preciso rodar as etapas 1 e 2 de novo.
- **A IA pode errar**, e esta é menor que o ChatGPT e o Gemini. Confira sempre os artigos citados: o texto oficial prevalece.
- **Não digite dados de processos ou de pessoas.**

Iniciativa independente, sem vínculo oficial com o Tribunal. © 2026 Carlos Mendes de Sousa. Todos os direitos reservados, exceto quanto aos textos das normas, que são atos oficiais públicos.

In [ ]:
#@title 1. Preparar a IA (uma vez por sessão, 5 a 10 minutos)
MODELO = "gemma4:12b"  #@param ["gemma4:12b", "gemma4:e4b", "gemma4:e2b"]
import shutil, subprocess, time, urllib.request

if not shutil.which("nvidia-smi"):
    raise SystemExit("GPU não encontrada. Abra Ambiente de execução › Alterar o tipo de ambiente de execução, "
                     "escolha GPU T4, salve e rode esta etapa de novo.")

print("Instalando programas…")
subprocess.run("apt-get -qq install -y zstd > /dev/null", shell=True, check=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1", shell=True, check=True)
subprocess.run("pip -q install onnxruntime tokenizers huggingface_hub", shell=True, check=True)

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434", timeout=2)
        break
    except Exception:
        time.sleep(1)

print(f"Baixando a IA {MODELO} (alguns minutos)…")
subprocess.run(["ollama", "pull", MODELO], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("Pronto. Siga para a etapa 2.")

In [ ]:
#@title 2. Carregar as normas e preparar a busca
import json, math, os, re, unicodedata, urllib.request
from collections import Counter, defaultdict
import numpy as np
import onnxruntime as ort
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer

SITE = "https://carlos-mendes-maestria.github.io/oraculo-normativo/dados/"

def baixar(nome):
    with urllib.request.urlopen(SITE + nome, timeout=120) as r:
        return r.read()

print("Baixando a base de normas…")
info = json.loads(baixar("indice_info.json"))
normas = json.loads(baixar("normas.json"))
grupos = json.loads(baixar("sinonimos.json"))["grupos"]
trechos, partes = [], []
for p in range(len(info["partes"])):
    trechos += json.loads(baixar(f"trechos_{p}.json"))
    partes.append(np.frombuffer(baixar(f"vetores_{p}.i8"), dtype=np.int8))
V = np.concatenate(partes).reshape(len(trechos), 384).astype(np.float32) / 127

# ---------- busca por palavras (BM25), igual a do site
STOP = set("a o as os e de do da dos das em no na nos nas um uma uns umas por para pelo pela pelos pelas com sem "
           "que se ao aos ou ser como qual quais quem onde quando pode tem ter ha mais seu sua seus suas este esta "
           "isso isto esse essa me eu".split())

def dobrar(t):
    return "".join(c for c in unicodedata.normalize("NFD", t.lower()) if unicodedata.category(c) != "Mn")

def termos(t):
    out = []
    for w in re.findall(r"[a-z0-9]+", dobrar(t)):
        if len(w) < 2 or w in STOP:
            continue
        out.append(w[:-1] if len(w) > 4 and w.endswith("s") else w)
    return out

print("Montando o índice…")
inv, comp = defaultdict(list), np.zeros(len(trechos), dtype=np.float32)
for i, (n, rot, est, txt) in enumerate(trechos):
    doc = f"{normas[n]['nome']} {normas[n]['ementa'] if rot == 'Preâmbulo' else ''} {rot} {est} {txt}"
    c = Counter(termos(doc))
    comp[i] = sum(c.values())
    for w, k in c.items():
        inv[w].append((i, k))
media = float(comp.mean())
idf = {w: math.log(1 + (len(trechos) - len(p) + 0.5) / (len(p) + 0.5)) for w, p in inv.items()}

def bm25(consulta, k1=1.2, b=0.75):
    s = np.zeros(len(trechos), dtype=np.float32)
    for w in set(termos(consulta)):
        for i, n in inv.get(w, []):
            s[i] += idf[w] * n * (k1 + 1) / (n + k1 * (1 - b + b * comp[i] / media))
    return s

# ---------- busca por significado: o mesmo modelo usado para gerar os vetores
print("Carregando o modelo de busca…")
REPO = "Xenova/multilingual-e5-small"
token = Tokenizer.from_file(hf_hub_download(REPO, "tokenizer.json"))
token.enable_truncation(512)
sessao = ort.InferenceSession(hf_hub_download(REPO, "onnx/model_quantized.onnx"), providers=["CPUExecutionProvider"])
entradas = {e.name for e in sessao.get_inputs()}

def vetor(consulta):
    e = token.encode("query: " + consulta)
    ids, mask = np.array([e.ids], dtype=np.int64), np.array([e.attention_mask], dtype=np.int64)
    feed = {"input_ids": ids, "attention_mask": mask}
    if "token_type_ids" in entradas:
        feed["token_type_ids"] = np.zeros_like(ids)
    h = sessao.run(None, feed)[0]
    m = mask[..., None].astype(np.float32)
    v = ((h * m).sum(1) / m.sum(1))[0]
    return v / np.linalg.norm(v)

# ---------- dicionario de equivalencias
SINONIMOS = [(re.compile(r"\b(?:" + "|".join(re.escape(s) for s in g["se"]) + r")\b"), g["acrescenta"]) for g in grupos]

def expandir(consulta):
    f = dobrar(consulta)
    extra = [a for rx, a in SINONIMOS if rx.search(f)]
    return consulta + " " + " ".join(extra) if extra else consulta

# ---------- remissoes: "art. 81 do Regimento Interno", "art. 5º da Resolução nº 45/2016"
por_arquivo = {n["arquivo"]: i for i, n in enumerate(normas)}
por_chave = defaultdict(list)
for i, n in enumerate(normas):
    if str(n["numero"]).isdigit():
        por_chave[(n["tipo"], int(n["numero"]), str(n["ano"]))].append(i)
if "Resolucao_02_1987" in por_arquivo:
    por_chave[("Resolucoes", 2, "1987")] = [por_arquivo["Resolucao_02_1987"]]
artigos = defaultdict(lambda: defaultdict(list))
for i, (n, rot, _, _) in enumerate(trechos):
    m = re.match(r"Art\. (\d+)", rot)
    if m and len(artigos[n][m.group(1)]) < 2:
        artigos[n][m.group(1)].append(i)

RX_CITACAO = re.compile(
    r"\bart(?:igo)?s?\.?\s*(\d{1,4})[º°o]?(?:-[A-Z])?((?:\s*(?:,|e)\s*(?:art\.?\s*)?\d{1,4}[º°o]?)*)"
    r"(?:(?!\bart)[^.;\n]){0,60}?\b("
    r"d[ao]s?\s+regimento\s+interno(?:\s+d[ao]\s+corregedoria)?|"
    r"(?:desta|deste)\s+(?:resolu[çc][ãa]o|provimento|lei)|"
    r"d[ao]s?\s+(?:resolu[çc][ãa]o|provimento(?:\s+conjunto)?|lei\s+complementar|lei))"
    r"(?:\s+(?:tjpi|cgj|estadual|ordin[áa]ria))*\s*(?:n[º°o.]*\s*)?(\d{1,2}\.\d{3}|\d{1,4})?"
    r"(?:\s*/\s*(\d{2,4})|,?\s*de\s+\d{1,2}[º°o]?\s+de\s+[a-zç]+\s+de\s+(\d{4}))?", re.I)

def norma_citada(tipo, num, ano, origem):
    t = dobrar(tipo)
    if "regimento interno" in t:
        return por_arquivo.get("Provimento_21_2014" if "corregedoria" in t else "Resolucao_02_1987")
    if t.startswith(("desta", "deste")):
        return origem
    if not num:
        return None
    a = int(ano) if ano else None
    if a is not None and a < 100:
        a += 2000 if a < 50 else 1900
    t2 = re.sub(r"^d[ao]s?\s+", "", t)
    g = ("Resolucoes" if t2.startswith("resolu") else "Provimentos" if t2.startswith("provimento")
         else "Leis Complementares" if "complementar" in t2 else "Leis Ordinarias")
    cands = [i for i in por_chave.get((g, int(num.replace(".", "")), str(a)), [])
             if ("conjunto" in t) == ("conjunto" in normas[i]["arquivo"].lower())]
    vig = [i for i in cands if normas[i]["situacao"] != "revogada"]
    return (vig or cands or [None])[0]

def citacoes(i):
    origem, _, _, texto = trechos[i]
    out = []
    for m in RX_CITACAO.finditer(texto):
        n = norma_citada(m.group(3), m.group(4), m.group(5) or m.group(6), origem)
        if n is None:
            continue
        for a in [m.group(1)] + re.findall(r"\d{1,4}", m.group(2) or ""):
            for t in artigos[n].get(str(int(a)), []):
                if t != i and t not in out:
                    out.append(t)
    return out

# ---------- busca hibrida (fusao das listas por RRF)
def rrf(listas, pesos, k=60):
    s = defaultdict(float)
    for l, p in zip(listas, pesos):
        for pos, i in enumerate(l):
            s[i] += p / (k + pos + 1)
    return sorted(s, key=lambda i: -s[i])

def buscar(consulta, so_vigentes=True, limite=30, por_norma=3):
    valido = lambda i: not so_vigentes or normas[trechos[i][0]]["situacao"] != "revogada"
    topo = lambda s: [int(i) for i in np.argsort(-s)[:400] if s[i] > 0 and valido(int(i))][:100]
    listas, pesos = [topo(bm25(consulta)), topo(V @ vetor(consulta) + 1)], [1, 1]
    amp = expandir(consulta)
    if amp != consulta:
        listas.append(topo(bm25(amp))); pesos.append(0.5)
    cont, res = Counter(), []
    for i in rrf(listas, pesos):
        n = trechos[i][0]
        if cont[n] < por_norma:
            cont[n] += 1
            res.append(i)
            if len(res) >= limite:
                break
    return res

print(f"Pronto: {len(normas):,} normas e {len(trechos):,} trechos. Siga para a etapa 3.".replace(",", "."))

In [ ]:
#@title 3. Consultar as normas
tema = "folga por plantão"  #@param {type:"string"}
pergunta = "com quantos dias de antecedência preciso pedir a folga?"  #@param {type:"string"}
so_normas_vigentes = True  #@param {type:"boolean"}
#@markdown Deixe a **pergunta** em branco para uma consulta geral sobre o tema. A resposta leva de 1 a 3 minutos.
import json, urllib.request

def escolher(res, limite, excluir=()):
    cont, out = Counter(), []
    for i in res:
        n = trechos[i][0]
        if i in excluir or cont[n] >= 2:
            continue
        cont[n] += 1
        out.append(i)
        if len(out) >= limite:
            break
    return out

def cabecalho(i):
    n, rot = normas[trechos[i][0]], trechos[i][1]
    sit = "revogada" if n["situacao"] == "revogada" else "vigente"
    por = f" pela {n['revogada_por']}" if n.get("revogada_por") else ""
    data = f", {n['data']}" if n.get("data") else ""
    return f"{n['nome']} – {rot} ({sit}{por}{data})"

def montar_prompt(tema, pergunta, so_vigentes):
    bloco_a = escolher(buscar(tema, so_vigentes), 8 if pergunta else 14)
    bloco_b = escolher(buscar(pergunta, so_vigentes), 8, set(bloco_a)) if pergunta else []
    todos = bloco_a + bloco_b
    citados = []
    for i in todos:
        for t in citacoes(i):
            if t not in todos and t not in citados:
                citados.append(t)
    citados = citados[:8]
    texto = lambda l, ini: "\n\n".join(f"[{ini + k + 1}] {cabecalho(i)}\n{trechos[i][3]}" for k, i in enumerate(l))
    ident = f"Tema: {tema} | Pergunta: {pergunta}" if pergunta else f"Tema: {tema} | Consulta geral, sem pergunta específica"
    regras = ["- Use SOMENTE os trechos abaixo e cite a norma e o artigo de cada informação, no formato (Resolução nº X/AAAA, art. Y).",
              "- Regras gerais dos servidores ou dos magistrados (ex.: Estatuto dos Servidores, Constituição Estadual) se aplicam a todos os cargos da categoria, mesmo que o trecho não cite o cargo perguntado.",
              "- Se houver norma revogada ou redações diferentes do mesmo artigo, use a vigente e a mais recente e mencione a diferença."]
    p = ["Você é um assistente que tira dúvidas sobre as normas internas do Poder Judiciário do Estado do Piauí.",
         "Responda em português, em linguagem clara, para um servidor sem formação jurídica, nesta ordem.",
         f'Logo abaixo do título de CADA parte da resposta, escreva a linha de identificação: "{ident}".', "",
         f'PARTE 1 – VISÃO GERAL DO TEMA "{tema}" (com base nas normas do acervo)',
         "- Faça uma explicação minuciosa do que as normas abaixo dispõem sobre o tema: quais normas tratam dele, para quem se aplicam, as regras, requisitos, prazos, procedimentos, exceções e como as normas se relacionam entre si.",
         *regras, "- Não resuma: desenvolva cada ponto relevante encontrado nos trechos.", ""]
    if pergunta:
        p += [f'PARTE 2 – RESPOSTA À PERGUNTA "{pergunta}" (com base nas normas do acervo)',
              "- Responda à pergunta de forma completa e direta.", *regras,
              "- Se os trechos não bastarem, diga claramente o que ficou sem resposta.", ""]
    p += ["PARTE 3 – SOMENTE se a Parte 2 não responder por completo:" if pergunta
          else "PARTE 2 – SOMENTE se algum aspecto importante do tema não estiver coberto pelos trechos:",
          '- Abra uma seção separada com o título exato: "⚠️ ATENÇÃO – INFORMAÇÃO FORA DAS NORMAS DO ACERVO (CONFERIR)".',
          "- Nela, complemente com seu conhecimento geral (ex.: Constituição Federal, leis federais), indicando a fonte provável de cada informação.",
          "- Termine a seção avisando que esse conteúdo não veio das normas consultadas e precisa ser conferido no texto oficial antes de ser usado.",
          "- Nunca misture esse conteúdo nas partes anteriores. Se elas já bastarem, não escreva esta seção.", "",
          f"TEMA PESQUISADO: {tema}", f"PERGUNTA DO USUÁRIO: {pergunta or 'nenhuma (consulta geral sobre o tema)'}", "",
          "TRECHOS SOBRE O TEMA:", texto(bloco_a, 0), ""]
    if bloco_b:
        p += ["TRECHOS ENCONTRADOS PARA A PERGUNTA:", texto(bloco_b, len(bloco_a)), ""]
    if citados:
        p += ["TRECHOS CITADOS (artigos de normas mencionados nos trechos acima; use-os para completar a resposta quando o trecho remeter a eles):",
              "\n\n".join(f"[C{k + 1}] {cabecalho(i)}\n{trechos[i][3]}" for k, i in enumerate(citados)), ""]
    p.append("Lembrete: o texto oficial publicado prevalece.")
    return "\n".join(p), todos + citados

def responder(prompt, modelo):
    corpo = {"model": modelo, "stream": True, "think": False, "messages": [{"role": "user", "content": prompt}],
             "options": {"num_ctx": 16384, "num_predict": 3000, "temperature": 0.2}}
    req = urllib.request.Request("http://127.0.0.1:11434/api/chat", json.dumps(corpo).encode(),
                                 {"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=1800) as r:
        for linha in r:
            print(json.loads(linha).get("message", {}).get("content", ""), end="", flush=True)
    print()

if not tema.strip():
    raise SystemExit("Preencha o tema.")
prompt, usados = montar_prompt(tema.strip(), pergunta.strip(), so_normas_vigentes)
print("Consultando a IA. A resposta aparece aos poucos…\n")
responder(prompt, MODELO)
print("\n" + "-" * 70 + "\nNORMAS CONSULTADAS (confira no texto oficial):")
for i in usados:
    print(" •", cabecalho(i), "\n  ", normas[trechos[i][0]]["url"])